# Dinomaly + DINOv3 — MVTec AD 2 — Single-Class — ONE CATEGORY / RUN

Notebook Colab này chuyển implementation DINOv3 của fork [cnulab/Dinomaly](https://github.com/cnulab/Dinomaly) sang quy trình MVTec AD 2 giống notebook mẫu:

- Mỗi runtime chỉ train đúng một trong 8 category.
- Chỉ giải nén archive data/<CATEGORY>.tar.gz của category đang chọn.
- Backbone là DINOv3-B/16; model weights phải được tải hợp lệ từ Meta và đặt trên Google Drive.
- Checkpoint/resume theo số iteration để giảm rủi ro mất tiến trình khi Colab ngắt kết nối.
- Public test: lưu anomaly map, tính AU-PRO@0.05.
- Private/private-mixed: tùy chọn xuất TIFF float16 để đóng gói bằng code utils chính thức của MVTec AD 2.

Nguồn được pin ở commit 9535c38b1f2f4d1c1576a671323eed7bcd65e67c. Đây là bản chuyển đổi single-class cho MVTec AD 2, không phải notebook chính thức của tác giả.

Lưu ý quan trọng: DINOv3 weights có điều khoản cấp phép riêng. Notebook không tự tải weights và không nhúng link trực tiếp đến file weights. Hãy đăng ký/tải tại [Meta DINOv3 downloads](https://ai.meta.com/resources/models-and-libraries/dinov3-downloads/).

## 1. Mount Drive và chọn đúng 1 category

Chỉ sửa ba vị trí nếu cần:

1. PROJECT_DRIVE
2. CATEGORY
3. DINOV3_WEIGHT_DRIVE

Khuyến nghị Runtime → Change runtime type → GPU. DINOv3-B/16 là lựa chọn mặc định để phù hợp Colab T4/L4. Nếu CUDA OOM, giảm TRAIN_BATCH từ 8 xuống 4 hoặc 2, restart runtime rồi Run all; checkpoint sẽ resume.

In [4]:
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

from pathlib import Path, PurePosixPath
import os
import sys
import json
import shutil
import subprocess
import hashlib
import tarfile
import random
import time

# ============================================================
# PROJECT PATH
# ============================================================

PROJECT_DRIVE = Path(
    "/content/drive/MyDrive/[Q3-4] 2026/[S7] Computer Vision/CV-Nhóm 9"
)

ARCHIVE_DIR = PROJECT_DRIVE / "data"
DRIVE_OUT = PROJECT_DRIVE / "dinomaly_dinov3_results_single_class"

# Tải hợp lệ từ Meta rồi đặt file này trên Drive.
DINOV3_WEIGHT_DRIVE = (
    PROJECT_DRIVE
    / "weights"
    / "dinov3_vitb16_pretrain_lvd1689m-73cec8be.pth"
)

# ============================================================
# CHỌN ĐÚNG 1 CATEGORY / 1 LẦN CHẠY
# ============================================================

# CATEGORY = "can"
# CATEGORY = "fabric"
# CATEGORY = "fruit_jelly"
# CATEGORY = "rice"
# CATEGORY = "sheet_metal"
# CATEGORY = "vial"
CATEGORY = "wallplugs"
# CATEGORY = "walnuts"

ALL_CATEGORIES = [
    "can",
    "fabric",
    "fruit_jelly",
    "rice",
    "sheet_metal",
    "vial",
    "wallplugs",
    "walnuts",
]

assert CATEGORY in ALL_CATEGORIES, CATEGORY
CATEGORIES = [CATEGORY]  # luôn chỉ có đúng một phần tử

# ============================================================
# SOURCE / LOCAL DATA
# ============================================================

REPO = Path("/content/Dinomaly")
COMMIT = "9535c38b1f2f4d1c1576a671323eed7bcd65e67c"

MAD2_NATIVE_ROOT = REPO / "data" / "mvtec_ad_2"
DINOMALY_DATA_ROOT = REPO / "data" / "mvtec_ad_2_dinomaly"

# ============================================================
# DINOv3-B/16 SINGLE-CLASS CONFIG
# ============================================================

BACKBONE = "dinov3_vitb16"
IMAGE_SIZE = 512
CROP_SIZE = 448        # 448 chia hết cho patch size 16
TOTAL_ITERS = 5_000    # cấu hình class-separated của Dinomaly
TRAIN_BATCH = 8        # T4 OOM: giảm còn 4 hoặc 2
EVAL_BATCH = 4
NUM_WORKERS = 0        # ổn định hơn trên Colab
SEED = 1

SAVE_EVERY_STEPS = 250
LOG_EVERY_STEPS = 50
TOPK_RATIO = 0.01

# ============================================================
# OPTIONAL FLAGS
# ============================================================

FORCE_EXTRACT = False
FORCE_TRAIN = False
FORCE_PUBLIC_INFERENCE = True
SAVE_PUBLIC_FULLRES = True
RUN_PRIVATE_INFERENCE = True

assert ARCHIVE_DIR.is_dir(), ARCHIVE_DIR
DRIVE_OUT.mkdir(parents=True, exist_ok=True)

print("=" * 72)
print("SELECTED CATEGORY :", CATEGORY)
print("ARCHIVE_DIR       :", ARCHIVE_DIR)
print("DINOv3 WEIGHT     :", DINOV3_WEIGHT_DRIVE)
print("OUTPUT             :", DRIVE_OUT)
print("=" * 72)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
SELECTED CATEGORY : wallplugs
ARCHIVE_DIR       : /content/drive/MyDrive/[Q3-4] 2026/[S7] Computer Vision/CV-Nhóm 9/data
DINOv3 WEIGHT     : /content/drive/MyDrive/[Q3-4] 2026/[S7] Computer Vision/CV-Nhóm 9/weights/dinov3_vitb16_pretrain_lvd1689m-73cec8be.pth
OUTPUT             : /content/drive/MyDrive/[Q3-4] 2026/[S7] Computer Vision/CV-Nhóm 9/dinomaly_dinov3_results_single_class


## 2. GPU audit, clone đúng fork DINOv3 và cài dependency

Không cài nguyên requirements.txt vì repo pin các bản NumPy/Pandas/Scikit-learn quá cũ so với Colab hiện tại. Notebook giữ Torch/CUDA sẵn có của Colab và chỉ cài các dependency Python cần thiết.

Yêu cầu của fork: Python trên 3.10 và PyTorch từ 2.7.0.

In [5]:
def run(cmd, cwd=None, env=None):
    cmd = [str(x) for x in cmd]
    print("$", " ".join(cmd))
    return subprocess.run(cmd, cwd=cwd, env=env, check=True)

run(["nvidia-smi"])

# Giữ clone hiện có nếu đúng commit; clone lại nếu khác.
repo_ok = False
if (REPO / ".git").is_dir():
    try:
        current = subprocess.check_output(
            ["git", "rev-parse", "HEAD"],
            cwd=REPO,
            text=True,
        ).strip()
        repo_ok = current == COMMIT
    except Exception:
        repo_ok = False

if REPO.exists() and not repo_ok:
    shutil.rmtree(REPO)

if not repo_ok:
    run([
        "git",
        "clone",
        "https://github.com/cnulab/Dinomaly.git",
        REPO,
    ])
    run([
        "git",
        "checkout",
        "--detach",
        COMMIT,
    ], cwd=REPO)

run([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "timm==0.9.12",
    "scikit-image>=0.21",
    "scikit-learn>=1.3",
    "scipy>=1.10",
    "pandas>=2.0",
    "opencv-python-headless>=4.8",
    "matplotlib>=3.7",
    "tifffile>=2023.7",
    "tqdm>=4.64",
])

actual_commit = subprocess.check_output(
    ["git", "rev-parse", "HEAD"],
    cwd=REPO,
    text=True,
).strip()

assert actual_commit == COMMIT, (actual_commit, COMMIT)
assert DINOV3_WEIGHT_DRIVE.is_file(), (
    "Thiếu DINOv3 weights. Tải dinov3_vitb16_pretrain_lvd1689m-73cec8be.pth "
    f"từ Meta và đặt tại:\n{DINOV3_WEIGHT_DRIVE}"
)

import torch
from packaging.version import Version

torch_base_version = torch.__version__.split("+")[0]
assert Version(torch_base_version) >= Version("2.7.0"), torch.__version__
assert sys.version_info >= (3, 10), sys.version
assert torch.cuda.is_available(), "Hãy bật GPU runtime trước khi chạy."

print("REPO COMMIT :", actual_commit)
print("Python      :", sys.version.split()[0])
print("Torch       :", torch.__version__)
print("CUDA        :", torch.version.cuda)
print("GPU         :", torch.cuda.get_device_name(0))
print("Weight MB   :", round(DINOV3_WEIGHT_DRIVE.stat().st_size / 2**20, 2))

$ nvidia-smi
$ /usr/bin/python3 -m pip install -q timm==0.9.12 scikit-image>=0.21 scikit-learn>=1.3 scipy>=1.10 pandas>=2.0 opencv-python-headless>=4.8 matplotlib>=3.7 tifffile>=2023.7 tqdm>=4.64
REPO COMMIT : 9535c38b1f2f4d1c1576a671323eed7bcd65e67c
Python      : 3.13.15
Torch       : 2.11.0+cu128
CUDA        : 12.8
GPU         : NVIDIA L4
Weight MB   : 326.98


## 3. Giải nén chỉ category đang chọn

Notebook tìm archive:

data/<CATEGORY>.tar.gz

và chỉ lấy bốn split cần cho workflow này: train, test_public, test_private, test_private_mixed. Dữ liệu của bảy category còn lại không được giải nén.

In [6]:
SPLITS_NEEDED = {
    "train",
    "test_public",
    "test_private",
    "test_private_mixed",
}

archive = ARCHIVE_DIR / f"{CATEGORY}.tar.gz"
assert archive.is_file(), f"Không tìm thấy archive:\n{archive}"

category_native_root = MAD2_NATIVE_ROOT / CATEGORY

def split_has_png(root: Path, split: str):
    split_root = root / split
    return split_root.exists() and any(split_root.rglob("*.png"))

def extract_selected_splits(archive_path, destination, splits, force=False):
    destination.mkdir(parents=True, exist_ok=True)

    if not force and all(split_has_png(destination, s) for s in splits):
        print("[SKIP EXTRACT] Đã có đủ split:", destination)
        return

    for split in splits:
        shutil.rmtree(destination / split, ignore_errors=True)

    copied = {split: 0 for split in splits}

    with tarfile.open(archive_path, "r:gz") as tf:
        for member in tf.getmembers():
            parts = PurePosixPath(member.name).parts
            positions = [i for i, part in enumerate(parts) if part in splits]
            if not positions:
                continue

            i = positions[0]
            split = parts[i]
            rel_parts = parts[i:]

            if any(part in ("", ".", "..") for part in rel_parts):
                raise RuntimeError(f"Unsafe archive path: {member.name}")

            target = destination.joinpath(*rel_parts)

            if member.isdir():
                target.mkdir(parents=True, exist_ok=True)
                continue
            if not member.isfile():
                continue

            target.parent.mkdir(parents=True, exist_ok=True)
            src = tf.extractfile(member)
            if src is None:
                raise RuntimeError(member.name)

            with src, open(target, "wb") as dst:
                shutil.copyfileobj(src, dst, length=8 * 1024 * 1024)
            copied[split] += 1

    for split in splits:
        assert split_has_png(destination, split), f"Extract lỗi: {CATEGORY}/{split}"

    print("Extract counts:", copied)

extract_selected_splits(
    archive,
    category_native_root,
    SPLITS_NEEDED,
    force=FORCE_EXTRACT,
)

checks = {
    "train/good": len(list((category_native_root / "train" / "good").glob("*.png"))),
    "test_public/good": len(list((category_native_root / "test_public" / "good").glob("*.png"))),
    "test_public/bad": len(list((category_native_root / "test_public" / "bad").glob("*.png"))),
    "test_public/ground_truth/bad": len(list(
        (category_native_root / "test_public" / "ground_truth" / "bad").glob("*_mask.png")
    )),
    "test_private": len(list((category_native_root / "test_private").rglob("*.png"))),
    "test_private_mixed": len(list((category_native_root / "test_private_mixed").rglob("*.png"))),
}

for key, value in checks.items():
    print(f"{key:35s}: {value}")
    assert value > 0, f"{CATEGORY}: thiếu {key}"

print("DATASET AUDIT PASS:", CATEGORY)

Extract counts: {'train': 293, 'test_private': 232, 'test_private_mixed': 232, 'test_public': 240}
train/good                         : 293
test_public/good                   : 60
test_public/bad                    : 90
test_public/ground_truth/bad       : 90
test_private                       : 232
test_private_mixed                 : 232
DATASET AUDIT PASS: wallplugs


## 4. Tạo staging schema tương thích MVTecDataset của Dinomaly

Repo cần schema:

<CATEGORY>/
  train/good/
  test/good/
  test/bad/
  ground_truth/bad/

MVTec AD 2 dùng test_public, vì vậy cell này tạo symlink, không nhân đôi ảnh.

In [7]:
def link_dir(src: Path, dst: Path):
    assert src.is_dir(), src

    if dst.is_symlink() or dst.is_file():
        dst.unlink()
    elif dst.exists():
        shutil.rmtree(dst)

    dst.parent.mkdir(parents=True, exist_ok=True)
    os.symlink(src, dst, target_is_directory=True)

staged_root = DINOMALY_DATA_ROOT / CATEGORY

if staged_root.exists() or staged_root.is_symlink():
    if staged_root.is_symlink():
        staged_root.unlink()
    else:
        shutil.rmtree(staged_root)

native = MAD2_NATIVE_ROOT / CATEGORY

link_dir(native / "train" / "good", staged_root / "train" / "good")
link_dir(native / "test_public" / "good", staged_root / "test" / "good")
link_dir(native / "test_public" / "bad", staged_root / "test" / "bad")
link_dir(
    native / "test_public" / "ground_truth" / "bad",
    staged_root / "ground_truth" / "bad",
)

assert list((staged_root / "train" / "good").glob("*.png"))
assert list((staged_root / "test" / "good").glob("*.png"))
assert list((staged_root / "test" / "bad").glob("*.png"))
assert list((staged_root / "ground_truth" / "bad").glob("*_mask.png"))

print("STAGING PASS:", staged_root)

STAGING PASS: /content/Dinomaly/data/mvtec_ad_2_dinomaly/wallplugs


## 5. Import module và build Dinomaly DINOv3-B/16

Các lựa chọn bám fork DINOv3:

- target layers 2–9
- hai nhóm fusion 0–3 và 4–7
- một bottleneck MLP, tám decoder blocks
- StableAdamW, cosine hard-mining loss
- Resize 512 → CenterCrop 448 → ImageNet normalize

Hai thay đổi an toàn:

- Freeze backbone thật sự bằng requires_grad=False để không giữ gradient DINOv3.
- Thêm epsilon 1e-6 vào LinearAttention2 theo khuyến nghị xử lý NaN của Dinomaly.

In [8]:
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import cv2
import tifffile
import matplotlib.pyplot as plt

from functools import partial
from tqdm.auto import tqdm
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision.datasets import ImageFolder

from dataset import MVTecDataset, get_data_transforms
from dinov1.utils import trunc_normal_
from dinov3.hub.backbones import load_dinov3_model
from models.uad import ViTill
from models.vision_transformer import Block as VitBlock, bMlp
from optimizers import StableAdamW
from utils import (
    WarmCosineScheduler,
    global_cosine_hm_percent,
    cal_anomaly_maps,
    get_gaussian_kernel,
)

device = torch.device("cuda:0")

def setup_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

setup_seed(SEED)

data_transform, gt_transform = get_data_transforms(
    IMAGE_SIZE,
    CROP_SIZE,
)

class LinearAttention2Safe(nn.Module):
    """LinearAttention2 của Dinomaly với epsilon chống chia cho 0."""

    def __init__(
        self,
        dim,
        num_heads=8,
        qkv_bias=False,
        qk_scale=None,
        attn_drop=0.0,
        proj_drop=0.0,
        eps=1e-6,
    ):
        super().__init__()
        self.num_heads = num_heads
        self.eps = eps
        head_dim = dim // num_heads
        self.scale = qk_scale or head_dim ** -0.5
        self.qkv = nn.Linear(dim, dim * 3, bias=qkv_bias)
        self.attn_drop = nn.Dropout(attn_drop)
        self.proj = nn.Linear(dim, dim)
        self.proj_drop = nn.Dropout(proj_drop)

    def forward(self, x, attn_mask=None):
        batch, tokens, channels = x.shape
        qkv = (
            self.qkv(x)
            .reshape(batch, tokens, 3, self.num_heads, channels // self.num_heads)
            .permute(2, 0, 3, 1, 4)
        )
        q, k, v = qkv[0], qkv[1], qkv[2]
        q = F.elu(q) + 1.0
        k = F.elu(k) + 1.0

        kv = torch.einsum("...sd,...se->...de", k, v)
        denom = torch.einsum("...sd,...d->...s", q, k.sum(dim=-2))
        z = 1.0 / (denom + self.eps)
        x = torch.einsum("...de,...sd,...s->...se", kv, q, z)
        x = x.transpose(1, 2).reshape(batch, tokens, channels)
        x = self.proj_drop(self.proj(x))
        return x, kv

def build_model():
    target_layers = [2, 3, 4, 5, 6, 7, 8, 9]
    fuse_layer_encoder = [[0, 1, 2, 3], [4, 5, 6, 7]]
    fuse_layer_decoder = [[0, 1, 2, 3], [4, 5, 6, 7]]

    encoder = load_dinov3_model(
        BACKBONE,
        layers_to_extract_from=target_layers,
        pretrained_weight_path=str(DINOV3_WEIGHT_DRIVE),
    )

    # Backbone chỉ làm teacher/frozen feature extractor.
    encoder.eval()
    for parameter in encoder.parameters():
        parameter.requires_grad_(False)

    embed_dim, num_heads = 768, 12

    bottleneck = nn.ModuleList([
        bMlp(embed_dim, embed_dim * 4, embed_dim, drop=0.2)
    ])

    decoder = nn.ModuleList([
        VitBlock(
            dim=embed_dim,
            num_heads=num_heads,
            mlp_ratio=4.0,
            qkv_bias=True,
            norm_layer=partial(nn.LayerNorm, eps=1e-8),
            attn=LinearAttention2Safe,
        )
        for _ in range(8)
    ])

    model = ViTill(
        encoder=encoder,
        bottleneck=bottleneck,
        decoder=decoder,
        target_layers=target_layers,
        mask_neighbor_size=0,
        fuse_layer_encoder=fuse_layer_encoder,
        fuse_layer_decoder=fuse_layer_decoder,
    )

    trainable = nn.ModuleList([bottleneck, decoder])

    for module in trainable.modules():
        if isinstance(module, nn.Linear):
            trunc_normal_(module.weight, std=0.01, a=-0.03, b=0.03)
            if module.bias is not None:
                nn.init.constant_(module.bias, 0)
        elif isinstance(module, nn.LayerNorm):
            nn.init.constant_(module.bias, 0)
            nn.init.constant_(module.weight, 1.0)

    model = model.to(device)
    return model, trainable

print("GPU       :", torch.cuda.get_device_name(0))
print("Backbone  :", BACKBONE)
print("Preprocess: Resize 512 -> CenterCrop 448")
print("Setting   : single-class / one category per runtime")

GPU       : NVIDIA L4
Backbone  : dinov3_vitb16
Preprocess: Resize 512 -> CenterCrop 448
Setting   : single-class / one category per runtime


## 6. Train selected category với checkpoint/resume

Cell này không có vòng lặp qua 8 category. Chỉ CATEGORY được train.

Checkpoint last.pth có optimizer để resume. Checkpoint model.pth chỉ chứa bottleneck + decoder và cấu hình; backbone luôn nạp lại từ file DINOv3 gốc. Việc này tránh lưu lặp hàng trăm MB weights frozen trong mỗi category.

In [9]:
SAVE_NAME = (
    "Dinomaly-Single-Class"
    "_dataset=MVTec-AD2"
    "_Encoder=dinov3_vitb16"
    f"_Resize={IMAGE_SIZE}"
    f"_Crop={CROP_SIZE}"
    f"_Iters={TOTAL_ITERS}"
)

CHECKPOINT_ROOT = DRIVE_OUT / "saved_results" / SAVE_NAME
CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)

def atomic_torch_save(payload, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    torch.save(payload, tmp)
    os.replace(tmp, path)

def train_selected_category(category, force=False):
    out_dir = CHECKPOINT_ROOT / category
    out_dir.mkdir(parents=True, exist_ok=True)

    final_ckpt = out_dir / "model.pth"
    resume_ckpt = out_dir / "last.pth"
    history_path = out_dir / "train_history.csv"

    if final_ckpt.is_file() and not force:
        print("[SKIP TRAIN] Final checkpoint đã tồn tại:", final_ckpt)
        return final_ckpt

    setup_seed(SEED)

    train_data = ImageFolder(
        root=str(DINOMALY_DATA_ROOT / category / "train"),
        transform=data_transform,
    )
    assert len(train_data) > 0

    generator = torch.Generator()
    generator.manual_seed(SEED)

    train_loader = DataLoader(
        train_data,
        batch_size=TRAIN_BATCH,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        drop_last=len(train_data) >= TRAIN_BATCH,
        generator=generator,
    )
    assert len(train_loader) > 0

    model, trainable = build_model()

    optimizer = StableAdamW(
        [{"params": trainable.parameters()}],
        lr=2e-3,
        betas=(0.9, 0.999),
        weight_decay=1e-4,
        amsgrad=True,
        eps=1e-10,
    )

    scheduler = WarmCosineScheduler(
        optimizer,
        base_value=2e-3,
        final_value=2e-4,
        total_iters=TOTAL_ITERS,
        warmup_iters=100,
    )

    global_step = 0
    epoch = 0
    history = []

    if history_path.is_file() and not force:
        try:
            history = pd.read_csv(history_path).to_dict(orient="records")
        except Exception as exc:
            print("[WARNING] Không đọc được history:", exc)

    if resume_ckpt.is_file() and not force:
        print("[LOAD RESUME]", resume_ckpt)
        state = torch.load(
            resume_ckpt,
            map_location="cpu",
            weights_only=False,
        )
        trainable.load_state_dict(state["trainable"], strict=True)
        optimizer.load_state_dict(state["optimizer"])
        scheduler.load_state_dict(state["scheduler"])
        global_step = int(state["global_step"])
        epoch = int(state.get("epoch", 0))
        print(f"[RESUME] step {global_step}/{TOTAL_ITERS}")

    def save_resume():
        atomic_torch_save(
            {
                "trainable": trainable.state_dict(),
                "optimizer": optimizer.state_dict(),
                "scheduler": scheduler.state_dict(),
                "global_step": global_step,
                "epoch": epoch,
                "category": category,
                "backbone": BACKBONE,
                "repo_commit": COMMIT,
            },
            resume_ckpt,
        )
        pd.DataFrame(history).to_csv(history_path, index=False)

    print("=" * 72)
    print("TRAIN CATEGORY :", category)
    print("Train images   :", len(train_data))
    print("Batch          :", TRAIN_BATCH)
    print("Batches/epoch  :", len(train_loader))
    print("Target steps   :", TOTAL_ITERS)
    print("Start step     :", global_step)
    print("Checkpoint     :", out_dir)
    print("=" * 72)

    window_losses = []
    started = time.time()

    try:
        while global_step < TOTAL_ITERS:
            model.train()
            model.encoder.eval()

            for images, _ in train_loader:
                if global_step >= TOTAL_ITERS:
                    break

                images = images.to(device, non_blocking=True)
                en, de = model(images)

                p = min(0.9 * global_step / 1000.0, 0.9)
                loss = global_cosine_hm_percent(
                    en,
                    de,
                    p=p,
                    factor=0.1,
                )

                if not torch.isfinite(loss):
                    save_resume()
                    raise RuntimeError(
                        f"Loss không hữu hạn tại step {global_step}: {loss.item()}"
                    )

                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                nn.utils.clip_grad_norm_(
                    trainable.parameters(),
                    max_norm=0.1,
                )
                optimizer.step()
                scheduler.step()

                global_step += 1
                window_losses.append(float(loss.detach().cpu()))

                if global_step % LOG_EVERY_STEPS == 0:
                    mean_loss = float(np.mean(window_losses))
                    elapsed_min = (time.time() - started) / 60.0
                    lr = float(optimizer.param_groups[0]["lr"])
                    history.append({
                        "step": global_step,
                        "loss": mean_loss,
                        "lr": lr,
                        "elapsed_min": elapsed_min,
                    })
                    print(
                        f"step {global_step:5d}/{TOTAL_ITERS} | "
                        f"loss {mean_loss:.6f} | lr {lr:.3e} | "
                        f"{elapsed_min:.1f} min"
                    )
                    window_losses = []

                if global_step % SAVE_EVERY_STEPS == 0:
                    save_resume()
                    print("[CHECKPOINT]", resume_ckpt)

            epoch += 1

    except KeyboardInterrupt:
        save_resume()
        print("[INTERRUPTED] Đã lưu checkpoint:", resume_ckpt)
        raise

    except RuntimeError as exc:
        if "out of memory" in str(exc).lower():
            try:
                save_resume()
            except Exception:
                pass
            torch.cuda.empty_cache()
            print(
                "CUDA OOM: giảm TRAIN_BATCH xuống 4 hoặc 2, "
                "restart runtime và Run all để resume."
            )
        raise

    final_payload = {
        "trainable": trainable.state_dict(),
        "global_step": global_step,
        "category": category,
        "backbone": BACKBONE,
        "image_size": IMAGE_SIZE,
        "crop_size": CROP_SIZE,
        "repo_commit": COMMIT,
        "seed": SEED,
    }
    atomic_torch_save(final_payload, final_ckpt)
    save_resume()

    print("[TRAIN COMPLETE]", final_ckpt)

    del model, trainable, optimizer, scheduler
    torch.cuda.empty_cache()

    return final_ckpt

trained_checkpoint = train_selected_category(
    CATEGORY,
    force=FORCE_TRAIN,
)

print("TRAIN DONE:", CATEGORY)
print("Checkpoint:", trained_checkpoint)

<All keys matched successfully>
TRAIN CATEGORY : wallplugs
Train images   : 293
Batch          : 8
Batches/epoch  : 36
Target steps   : 5000
Start step     : 0
Checkpoint     : /content/drive/MyDrive/[Q3-4] 2026/[S7] Computer Vision/CV-Nhóm 9/dinomaly_dinov3_results_single_class/saved_results/Dinomaly-Single-Class_dataset=MVTec-AD2_Encoder=dinov3_vitb16_Resize=512_Crop=448_Iters=5000/wallplugs
step    50/5000 | loss 0.274259 | lr 1.010e-03 | 1.1 min
step   100/5000 | loss 0.093270 | lr 2.000e-03 | 2.1 min
step   150/5000 | loss 0.069134 | lr 2.000e-03 | 3.1 min
step   200/5000 | loss 0.060106 | lr 1.998e-03 | 4.2 min
step   250/5000 | loss 0.053546 | lr 1.996e-03 | 5.2 min
[CHECKPOINT] /content/drive/MyDrive/[Q3-4] 2026/[S7] Computer Vision/CV-Nhóm 9/dinomaly_dinov3_results_single_class/saved_results/Dinomaly-Single-Class_dataset=MVTec-AD2_Encoder=dinov3_vitb16_Resize=512_Crop=448_Iters=5000/wallplugs/last.pth
step   300/5000 | loss 0.049719 | lr 1.993e-03 | 6.4 min
step   350/5000 |

## 7. Public inference — chỉ selected category

Anomaly map được tính bằng cosine distance trung bình từ hai cặp feature fusion, sau đó Gaussian smoothing giống evaluation của repo.

Mặc định chỉ lưu map 448×448 để tiết kiệm Drive. Bật SAVE_PUBLIC_FULLRES nếu cần map public ở độ phân giải gốc.

In [10]:
RAW_ROOT = DRIVE_OUT / "raw_maps"
CROP_ROOT = RAW_ROOT / "official_crop"
FULL_ROOT = RAW_ROOT / "fullres"
PRIVATE_TIFF_ROOT = DRIVE_OUT / "anomaly_images"

def load_trained_model(category):
    checkpoint = CHECKPOINT_ROOT / category / "model.pth"
    assert checkpoint.is_file(), checkpoint

    model, trainable = build_model()
    state = torch.load(
        checkpoint,
        map_location="cpu",
        weights_only=False,
    )
    assert state["category"] == category
    assert state["backbone"] == BACKBONE
    trainable.load_state_dict(state["trainable"], strict=True)

    model.eval()
    model.encoder.eval()
    return model

@torch.inference_mode()
def predict_crop_maps(model, images):
    images = images.to(device, non_blocking=True)
    en, de = model(images)
    anomaly_maps, _ = cal_anomaly_maps(
        en,
        de,
        out_size=CROP_SIZE,
    )
    gaussian = get_gaussian_kernel(
        kernel_size=5,
        sigma=4,
    ).to(device)
    anomaly_maps = gaussian(anomaly_maps)
    return anomaly_maps[:, 0].float().cpu()

def export_public_selected(category, force=False):
    dataset = MVTecDataset(
        root=str(DINOMALY_DATA_ROOT / category),
        transform=data_transform,
        gt_transform=gt_transform,
        phase="test",
    )

    loader = DataLoader(
        dataset,
        batch_size=EVAL_BATCH,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True,
    )

    manifest_path = (
        DRIVE_OUT
        / "manifests"
        / f"{category}_test_public.csv"
    )
    manifest_path.parent.mkdir(parents=True, exist_ok=True)

    if manifest_path.is_file() and not force:
        previous = pd.read_csv(manifest_path)
        map_paths = [Path(p) for p in previous["crop_npy"].tolist()]
        if len(previous) == len(dataset) and all(p.is_file() for p in map_paths):
            print("[SKIP PUBLIC] Đã đủ anomaly maps:", manifest_path)
            return previous

    model = load_trained_model(category)
    rows = []

    for images, _, labels, image_paths in tqdm(
        loader,
        desc=f"public/{category}",
    ):
        batch_maps = predict_crop_maps(model, images).numpy()

        for anomaly_map, label, image_path in zip(
            batch_maps,
            labels.tolist(),
            image_paths,
        ):
            src = Path(image_path)
            kind = src.parent.name

            crop_dst = (
                CROP_ROOT
                / category
                / kind
                / f"{src.stem}.npy"
            )
            crop_dst.parent.mkdir(parents=True, exist_ok=True)
            np.save(crop_dst, anomaly_map.astype(np.float32))

            full_dst_value = None
            if SAVE_PUBLIC_FULLRES:
                with Image.open(src) as image:
                    width, height = image.size

                full_map = cv2.resize(
                    anomaly_map,
                    (width, height),
                    interpolation=cv2.INTER_LINEAR,
                ).astype(np.float32)

                full_dst = (
                    FULL_ROOT
                    / category
                    / kind
                    / f"{src.stem}.npy"
                )
                full_dst.parent.mkdir(parents=True, exist_ok=True)
                np.save(full_dst, full_map)
                full_dst_value = str(full_dst)

            flat = anomaly_map.reshape(-1)
            k = max(1, int(flat.size * TOPK_RATIO))
            image_score = float(np.partition(flat, -k)[-k:].mean())

            rows.append({
                "category": category,
                "kind": kind,
                "label": int(label),
                "image": str(src),
                "crop_npy": str(crop_dst),
                "fullres_npy": full_dst_value,
                "image_score_top1pct": image_score,
                "score_min": float(anomaly_map.min()),
                "score_max": float(anomaly_map.max()),
            })

    manifest = pd.DataFrame(rows)
    manifest.to_csv(manifest_path, index=False)

    del model
    torch.cuda.empty_cache()

    print("PUBLIC INFERENCE COMPLETE:", category)
    print("Images  :", len(manifest))
    print("Manifest:", manifest_path)
    return manifest

public_manifest = export_public_selected(
    CATEGORY,
    force=FORCE_PUBLIC_INFERENCE,
)

display(public_manifest.head())

<All keys matched successfully>


public/wallplugs:   0%|          | 0/38 [00:00<?, ?it/s]

PUBLIC INFERENCE COMPLETE: wallplugs
Images  : 150
Manifest: /content/drive/MyDrive/[Q3-4] 2026/[S7] Computer Vision/CV-Nhóm 9/dinomaly_dinov3_results_single_class/manifests/wallplugs_test_public.csv


,category,kind,label,image,crop_npy,fullres_npy,image_score_top1pct,score_min,score_max
0,wallplugs,good,0,/content/Dinomaly/data/mvtec_ad_2_dinomaly/wal...,/content/drive/MyDrive/[Q3-4] 2026/[S7] Comput...,/content/drive/MyDrive/[Q3-4] 2026/[S7] Comput...,0.096062,0.013679,0.119253
1,wallplugs,good,0,/content/Dinomaly/data/mvtec_ad_2_dinomaly/wal...,/content/drive/MyDrive/[Q3-4] 2026/[S7] Comput...,/content/drive/MyDrive/[Q3-4] 2026/[S7] Comput...,0.065105,0.005766,0.089206
2,wallplugs,good,0,/content/Dinomaly/data/mvtec_ad_2_dinomaly/wal...,/content/drive/MyDrive/[Q3-4] 2026/[S7] Comput...,/content/drive/MyDrive/[Q3-4] 2026/[S7] Comput...,0.062649,0.005499,0.097933
3,wallplugs,good,0,/content/Dinomaly/data/mvtec_ad_2_dinomaly/wal...,/content/drive/MyDrive/[Q3-4] 2026/[S7] Comput...,/content/drive/MyDrive/[Q3-4] 2026/[S7] Comput...,0.124805,0.014869,0.141304
4,wallplugs,good,0,/content/Dinomaly/data/mvtec_ad_2_dinomaly/wal...,/content/drive/MyDrive/[Q3-4] 2026/[S7] Comput...,/content/drive/MyDrive/[Q3-4] 2026/[S7] Comput...,0.065764,0.006016,0.089793


## 8. Public AU-PRO@0.05 — selected category

MVTec AD 2 nhấn mạnh segmentation AU-PRO ở FPR tối đa 0.05. Mỗi connected component của ground truth có trọng số bằng nhau. Kết quả public chỉ dùng để ước lượng ban đầu; không dùng public ground truth để tune cho private test.

In [11]:
MAX_FPR = 0.05

def compute_aupro_005(normal_scores, region_scores):
    normal_scores = np.concatenate(normal_scores).astype(
        np.float32,
        copy=False,
    )

    region_scores = [
        np.asarray(scores, dtype=np.float32).reshape(-1)
        for scores in region_scores
        if np.asarray(scores).size > 0
    ]

    num_regions = len(region_scores)
    if num_regions == 0:
        raise RuntimeError("Không tìm thấy GT connected component.")
    if normal_scores.size == 0:
        raise RuntimeError("Không có normal pixels để tính FPR.")

    anomaly_scores = np.concatenate(region_scores)
    anomaly_weights = np.concatenate([
        np.full(
            scores.size,
            1.0 / (num_regions * scores.size),
            dtype=np.float64,
        )
        for scores in region_scores
    ])

    scores_all = np.concatenate([normal_scores, anomaly_scores])
    fpr_delta = np.concatenate([
        np.full(
            normal_scores.size,
            1.0 / normal_scores.size,
            dtype=np.float64,
        ),
        np.zeros(anomaly_scores.size, dtype=np.float64),
    ])
    pro_delta = np.concatenate([
        np.zeros(normal_scores.size, dtype=np.float64),
        anomaly_weights,
    ])

    order = np.argsort(scores_all, kind="mergesort")[::-1]
    sorted_scores = scores_all[order]
    cumulative_fpr = np.cumsum(fpr_delta[order])
    cumulative_pro = np.cumsum(pro_delta[order])

    group_ends = np.r_[
        np.flatnonzero(sorted_scores[1:] != sorted_scores[:-1]),
        sorted_scores.size - 1,
    ]

    fpr = np.r_[0.0, cumulative_fpr[group_ends]]
    pro = np.r_[0.0, cumulative_pro[group_ends]]

    unique_fpr, first_idx, counts = np.unique(
        fpr,
        return_index=True,
        return_counts=True,
    )
    unique_pro = pro[first_idx + counts - 1]

    if unique_fpr[-1] < MAX_FPR:
        raise RuntimeError("FPR curve không đạt 0.05.")

    idx = np.searchsorted(unique_fpr, MAX_FPR, side="left")

    if idx < len(unique_fpr) and np.isclose(unique_fpr[idx], MAX_FPR):
        x = unique_fpr[:idx + 1]
        y = unique_pro[:idx + 1]
    else:
        x0, x1 = unique_fpr[idx - 1], unique_fpr[idx]
        y0, y1 = unique_pro[idx - 1], unique_pro[idx]
        y_at_limit = y0 + (MAX_FPR - x0) * (y1 - y0) / (x1 - x0)
        x = np.r_[unique_fpr[:idx], MAX_FPR]
        y = np.r_[unique_pro[:idx], y_at_limit]

    return float(np.trapezoid(y, x) / MAX_FPR)

def evaluate_selected_category(category):
    dataset = MVTecDataset(
        root=str(DINOMALY_DATA_ROOT / category),
        transform=data_transform,
        gt_transform=gt_transform,
        phase="test",
    )

    normal_scores = []
    region_scores = []
    n_good = 0
    n_bad = 0

    for idx in tqdm(
        range(len(dataset)),
        desc=f"AU-PRO0.05/{category}",
    ):
        _, gt, label, image_path = dataset[idx]
        src = Path(image_path)
        kind = src.parent.name

        map_path = (
            CROP_ROOT
            / category
            / kind
            / f"{src.stem}.npy"
        )
        assert map_path.is_file(), f"Thiếu public map:\n{map_path}"

        anomaly_map = np.load(
            map_path,
            mmap_mode="r",
        ).astype(np.float32, copy=False)

        gt_np = (gt[0].numpy() > 0.5).astype(np.uint8)
        assert anomaly_map.shape == gt_np.shape, (
            src,
            anomaly_map.shape,
            gt_np.shape,
        )

        normal_scores.append(anomaly_map[gt_np == 0].reshape(-1))

        if gt_np.any():
            num_labels, labels = cv2.connectedComponents(
                gt_np,
                connectivity=8,
            )
            for region_id in range(1, num_labels):
                region_scores.append(
                    anomaly_map[labels == region_id].reshape(-1)
                )

        if int(label) == 0:
            n_good += 1
        else:
            n_bad += 1

    score = compute_aupro_005(normal_scores, region_scores)

    return {
        "Category": category,
        "AU_PRO_0.05": score,
        "AU_PRO_0.05_pct": score * 100.0,
        "good_images": n_good,
        "bad_images": n_bad,
        "gt_regions": len(region_scores),
        "backbone": BACKBONE,
        "crop_size": CROP_SIZE,
        "setting": "single-class",
    }

result = evaluate_selected_category(CATEGORY)
result_df = pd.DataFrame([result])

metric_dir = DRIVE_OUT / "metrics"
metric_dir.mkdir(parents=True, exist_ok=True)
metric_path = metric_dir / f"{CATEGORY}.csv"
result_df.to_csv(metric_path, index=False)

display(result_df)
print(f"{CATEGORY} AU-PRO@0.05 = {result['AU_PRO_0.05_pct']:.4f}%")
print("Saved:", metric_path)

AU-PRO0.05/wallplugs:   0%|          | 0/150 [00:00<?, ?it/s]

,Category,AU_PRO_0.05,AU_PRO_0.05_pct,good_images,bad_images,gt_regions,backbone,crop_size,setting
0,wallplugs,0.122347,12.234698,60,90,102,dinov3_vitb16,448,single-class


wallplugs AU-PRO@0.05 = 12.2347%
Saved: /content/drive/MyDrive/[Q3-4] 2026/[S7] Computer Vision/CV-Nhóm 9/dinomaly_dinov3_results_single_class/metrics/wallplugs.csv


## 9. Tổng hợp các category đã chạy — không dùng GPU

Cell này chỉ đọc metrics/<category>.csv đã tích lũy trên Drive. Nó không train batch. Khi đủ 8 category, macro mean được thêm vào bảng tổng.

In [12]:
metric_dir = DRIVE_OUT / "metrics"

available_rows = []
missing_categories = []

for category in ALL_CATEGORIES:
    path = metric_dir / f"{category}.csv"
    if path.is_file():
        one = pd.read_csv(path)
        assert len(one) == 1, (category, path)
        available_rows.append(one.iloc[0].to_dict())
    else:
        missing_categories.append(category)

summary_df = pd.DataFrame(available_rows)

if not summary_df.empty:
    order_map = {
        category: idx
        for idx, category in enumerate(ALL_CATEGORIES)
    }
    summary_df["_order"] = summary_df["Category"].map(order_map)
    summary_df = (
        summary_df
        .sort_values("_order")
        .drop(columns="_order")
        .reset_index(drop=True)
    )

display(summary_df)
print("Completed:", summary_df["Category"].tolist() if not summary_df.empty else [])
print("Missing  :", missing_categories)

if not missing_categories:
    final_df = summary_df.copy()
    macro = {
        "Category": "macro_mean",
        "AU_PRO_0.05": final_df["AU_PRO_0.05"].mean(),
        "AU_PRO_0.05_pct": final_df["AU_PRO_0.05_pct"].mean(),
        "good_images": final_df["good_images"].sum(),
        "bad_images": final_df["bad_images"].sum(),
        "gt_regions": final_df["gt_regions"].sum(),
        "backbone": BACKBONE,
        "crop_size": CROP_SIZE,
        "setting": "single-class",
    }
    final_df.loc[len(final_df)] = macro
    summary_path = DRIVE_OUT / "mvtec_ad2_dinomaly_dinov3_single_class.csv"
    final_df.to_csv(summary_path, index=False)
    print("8/8 COMPLETE:", summary_path)
    display(final_df)
else:
    partial_path = DRIVE_OUT / "mvtec_ad2_dinomaly_dinov3_partial.csv"
    summary_df.to_csv(partial_path, index=False)
    print("Partial table:", partial_path)

,Category,AU_PRO_0.05,AU_PRO_0.05_pct,good_images,bad_images,gt_regions,backbone,crop_size,setting
0,can,0.024728,2.472832,72,90,108,dinov3_vitb16,448,single-class
1,fabric,0.374399,37.439934,66,90,192,dinov3_vitb16,448,single-class
2,fruit_jelly,0.566003,56.600320,20,60,459,dinov3_vitb16,448,single-class
3,rice,0.489358,48.935775,42,90,114,dinov3_vitb16,448,single-class
4,sheet_metal,0.287096,28.709642,24,90,1677,dinov3_vitb16,448,single-class
5,vial,0.680287,68.028668,35,105,170,dinov3_vitb16,448,single-class
6,wallplugs,0.122347,12.234698,60,90,102,dinov3_vitb16,448,single-class


Completed: ['can', 'fabric', 'fruit_jelly', 'rice', 'sheet_metal', 'vial', 'wallplugs']
Missing  : ['walnuts']
Partial table: /content/drive/MyDrive/[Q3-4] 2026/[S7] Computer Vision/CV-Nhóm 9/dinomaly_dinov3_results_single_class/mvtec_ad2_dinomaly_dinov3_partial.csv


## 10. OPTIONAL — private / private-mixed inference

Mặc định RUN_PRIVATE_INFERENCE=False.

Khi bật, cell xuất continuous anomaly maps float16 tại:

anomaly_images/<CATEGORY>/<split>/<image>.tiff

Sau đó dùng code utils chính thức của MVTec AD 2 để kiểm tra đúng schema và đóng gói submission. Ground truth private không có cục bộ; điểm chính thức chỉ nhận được qua evaluation server.

In [13]:
class PrivateMAD2Dataset(Dataset):
    def __init__(self, root, category, split, transform):
        self.paths = sorted(
            (Path(root) / category / split).rglob("*.png")
        )
        assert self.paths, f"Không có ảnh: {root}/{category}/{split}"
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        path = self.paths[idx]
        with Image.open(path) as image:
            rgb = image.convert("RGB")
            width, height = rgb.size
            tensor = self.transform(rgb)
        return tensor, str(path), height, width

def export_private_selected(category, split):
    dataset = PrivateMAD2Dataset(
        MAD2_NATIVE_ROOT,
        category,
        split,
        data_transform,
    )
    loader = DataLoader(
        dataset,
        batch_size=1,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True,
    )

    model = load_trained_model(category)

    for image, image_path, height, width in tqdm(
        loader,
        desc=f"{category}/{split}",
    ):
        src = Path(image_path[0])
        height = int(height[0])
        width = int(width[0])

        tiff_dst = (
            PRIVATE_TIFF_ROOT
            / category
            / split
            / f"{src.stem}.tiff"
        )
        tiff_dst.parent.mkdir(parents=True, exist_ok=True)

        if tiff_dst.is_file():
            continue

        crop_map = predict_crop_maps(model, image)[0].numpy()
        full_map = cv2.resize(
            crop_map,
            (width, height),
            interpolation=cv2.INTER_LINEAR,
        ).astype(np.float32)

        tifffile.imwrite(
            tiff_dst,
            full_map.astype(np.float16),
        )

    del model
    torch.cuda.empty_cache()
    print("PRIVATE EXPORT COMPLETE:", category, split)

if RUN_PRIVATE_INFERENCE:
    for split in ["test_private", "test_private_mixed"]:
        export_private_selected(CATEGORY, split)
else:
    print(
        "SKIP PRIVATE. Đặt RUN_PRIVATE_INFERENCE=True ở Cell 1 "
        "sau khi public inference/metric đã PASS."
    )

<All keys matched successfully>


wallplugs/test_private:   0%|          | 0/232 [00:00<?, ?it/s]

PRIVATE EXPORT COMPLETE: wallplugs test_private
<All keys matched successfully>


wallplugs/test_private_mixed:   0%|          | 0/232 [00:00<?, ?it/s]

PRIVATE EXPORT COMPLETE: wallplugs test_private_mixed


## 11. Audit selected category

Audit ghi lại checkpoint hash, số map public, commit nguồn, backbone, weight filename và preprocess. Cell sẽ fail nếu train/inference/metric của category hiện tại chưa hoàn tất.

In [14]:
def sha256_file(path, chunk=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for data in iter(lambda: handle.read(chunk), b""):
            digest.update(data)
    return digest.hexdigest()

category_ckpt = CHECKPOINT_ROOT / CATEGORY / "model.pth"
category_metric = DRIVE_OUT / "metrics" / f"{CATEGORY}.csv"

public_good_maps = list(
    (CROP_ROOT / CATEGORY / "good").glob("*.npy")
)
public_bad_maps = list(
    (CROP_ROOT / CATEGORY / "bad").glob("*.npy")
)

expected_good = len(list(
    (MAD2_NATIVE_ROOT / CATEGORY / "test_public" / "good").glob("*.png")
))
expected_bad = len(list(
    (MAD2_NATIVE_ROOT / CATEGORY / "test_public" / "bad").glob("*.png")
))

assert category_ckpt.is_file(), category_ckpt
assert category_metric.is_file(), category_metric
assert len(public_good_maps) == expected_good
assert len(public_bad_maps) == expected_bad

audit = {
    "category": CATEGORY,
    "checkpoint": str(category_ckpt),
    "checkpoint_sha256": sha256_file(category_ckpt),
    "metric": str(category_metric),
    "public_good_expected": expected_good,
    "public_good_maps": len(public_good_maps),
    "public_bad_expected": expected_bad,
    "public_bad_maps": len(public_bad_maps),
    "repo_url": "https://github.com/cnulab/Dinomaly",
    "repo_commit": COMMIT,
    "backbone": BACKBONE,
    "backbone_weight_file": DINOV3_WEIGHT_DRIVE.name,
    "backbone_weight_size_bytes": DINOV3_WEIGHT_DRIVE.stat().st_size,
    "setting": "single-class",
    "total_iters": TOTAL_ITERS,
    "train_batch": TRAIN_BATCH,
    "preprocess": (
        "Resize 512 -> CenterCrop 448 -> ImageNet Normalize"
    ),
}

audit_dir = DRIVE_OUT / "audit"
audit_dir.mkdir(parents=True, exist_ok=True)
audit_path = audit_dir / f"{CATEGORY}.json"
audit_path.write_text(
    json.dumps(audit, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print(json.dumps(audit, indent=2, ensure_ascii=False))
print("AUDIT PASS:", audit_path)

{
  "category": "wallplugs",
  "checkpoint": "/content/drive/MyDrive/[Q3-4] 2026/[S7] Computer Vision/CV-Nhóm 9/dinomaly_dinov3_results_single_class/saved_results/Dinomaly-Single-Class_dataset=MVTec-AD2_Encoder=dinov3_vitb16_Resize=512_Crop=448_Iters=5000/wallplugs/model.pth",
  "checkpoint_sha256": "ab8f1c5889d711d50a0c7f7a5dbc871e91cdef0653e3134a2b81eb9b7034ef49",
  "metric": "/content/drive/MyDrive/[Q3-4] 2026/[S7] Computer Vision/CV-Nhóm 9/dinomaly_dinov3_results_single_class/metrics/wallplugs.csv",
  "public_good_expected": 60,
  "public_good_maps": 60,
  "public_bad_expected": 90,
  "public_bad_maps": 90,
  "repo_url": "https://github.com/cnulab/Dinomaly",
  "repo_commit": "9535c38b1f2f4d1c1576a671323eed7bcd65e67c",
  "backbone": "dinov3_vitb16",
  "backbone_weight_file": "dinov3_vitb16_pretrain_lvd1689m-73cec8be.pth",
  "backbone_weight_size_bytes": 342860279,
  "setting": "single-class",
  "total_iters": 5000,
  "train_batch": 8,
  "preprocess": "Resize 512 -> CenterCrop 448 

## Workflow chạy đủ 8 category

Không train batch. Lặp tám phiên độc lập:

1. Đổi CATEGORY.
2. Restart runtime.
3. Run all.
4. Xác nhận AUDIT PASS.
5. Chuyển sang category kế tiếp.

Thứ tự gợi ý:

can → fabric → fruit_jelly → rice → sheet_metal → vial → wallplugs → walnuts

Các checkpoint, metric và manifest được lưu trên Drive nên không mất khi runtime Colab bị reset. Nếu runtime ngắt giữa lúc train, Run all với cùng CATEGORY sẽ nạp last.pth và tiếp tục từ global_step gần nhất.